# BANKING77 model-selection analysis

This notebook reads a **completed, saved benchmark** and makes no model calls.
The default run is the 770-example stratified **training** cohort (10 examples per intent).
These are development results for inspecting behavior and choosing comparison models.
They are not final held-out thesis results.

Change `RUN_ID` below to analyze another completed BANKING77 run. Generated files belong in
`outputs/analysis/<run-id>/`; keep this source notebook cleared of outputs when versioning it.
Reusable code lives in `thesis_research.analysis`.

The figures show descriptive point estimates. Before final experiments, define the uncertainty
method, planned comparisons, and any correction for multiple comparisons.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from thesis_research.analysis import (
    confusion_pairs,
    export_analysis,
    load_benchmark,
    model_table,
    pairwise_table,
    plot_figures,
)

# Start the kernel in research/ or any directory underneath it.
RESEARCH_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "src/thesis_research").is_dir()
)
RUN_ID = "20260930T222653327742Z-d103c737bc91438482be7af965fb77be"
RUN_DIRECTORY = RESEARCH_ROOT / "outputs/benchmarks" / RUN_ID
OUTPUT_DIRECTORY = RESEARCH_ROOT / "outputs/analysis" / RUN_ID
analysis = load_benchmark(RUN_DIRECTORY)
print(analysis.caption)

## Run provenance and cohort

Use the frozen run configuration to describe the experiment. Validation checks that every model
covers the same example IDs and references, correctness and accuracy agree, and class support
and paired counts match the cohort.

In [ ]:
display(
    pd.Series(
        {
            "run": analysis.run.name,
            "name": analysis.metadata["name"],
            "status": analysis.metadata["status"],
            "started_at_utc": analysis.metadata["timestamp"],
            "dataset": analysis.cohort["dataset_id"],
            "revision": analysis.cohort["dataset_revision"],
            "split": analysis.cohort["split"],
            "examples": analysis.cohort["count"],
            "selection": analysis.cohort["selection"],
            "execution": analysis.metadata["plan"]["execution"],
            "llm_output": analysis.metadata["plan"]["llm_output"],
            "runner": analysis.metadata["runner"],
            "benchmark_code": analysis.metadata["code"],
        },
        name="value",
    )
)
cohort_support = (
    pd.Series([row["reference"]["label"] for row in analysis.cohort["examples"]])
    .value_counts()
    .sort_index()
)
print(
    f"Intents: {len(cohort_support)}; support range: {cohort_support.min()}–{cohort_support.max()}"
)

## Quality, latency, failures, and recorded cost

Accuracy includes all selected examples. A valid prediction can still be wrong. Macro-F1 weights
all 77 intents equally. Median and p95 latency use all measured decisions, including failures.
Timing reflects the runner, gateway, execution settings, model order, and session.

Costs come from recorded gateway usage. **Unavailable cost stays missing**, including Jev in the
default run. A cost comparison with Jev requires comparable cost data. Cost per 1,000 examples is
a linear scaling of this run's reported cost, not a current price quote.

In [ ]:
comparison = model_table(analysis)
display(
    comparison.style.format(
        {
            "accuracy": "{:.2%}",
            "macro_f1": "{:.4f}",
            "all_decision_latency_p50_seconds": "{:.3f}",
            "all_decision_latency_p95_seconds": "{:.3f}",
            "cost_usd": "${:.6f}",
            "cost_usd_per_1000_examples": "${:.6f}",
        },
        na_rep="unavailable",
    )
)

## Descriptive figures

The ECDF shows the fraction of measured decisions completed within each latency. Its log axis
preserves the slow tail. The intent heatmap uses the same 0–1 F1 scale for every model.
Each intent has only 10 examples in the default run, so intent-level scores are coarse.

In [ ]:
figures = plot_figures(analysis)
for name, figure in figures.items():
    print(name)
    display(figure)
    plt.close(figure)

## Jev–LLM complementarity

Paired correctness shows where only Jev succeeds, only the LLM succeeds, both succeed, or both
fail. It helps identify whether a combined method is worth investigating.

`oracle_accuracy_upper_bound` counts an example as correct if either model is correct. It uses
reference labels to choose the answer, so it is an **upper bound**, not a routing rule or a
measured combined-system result. The saved exact McNemar p-values are exploratory and uncorrected
for the three comparisons. They do not turn this training run into final evidence.

In [ ]:
pairs = pairwise_table(analysis)
display(
    pairs[
        [
            "baseline_model_id",
            "comparison_model_id",
            "total",
            "both_correct",
            "baseline_only_correct",
            "comparison_only_correct",
            "neither_correct",
            "accuracy_difference",
            "mcnemar_exact_two_sided_p_value",
            "oracle_accuracy_upper_bound",
        ]
    ].style.format(
        {
            "accuracy_difference": "{:+.2%}",
            "mcnemar_exact_two_sided_p_value": "{:.4g}",
            "oracle_accuracy_upper_bound": "{:.2%}",
        }
    )
)

## Error inspection

Start with recurring confusion pairs, then read the underlying messages. Failed or invalid
predictions remain in the metric denominator and can be inspected separately below.
Use development data for prompt changes and model selection. Reserve the test split for the
final frozen comparison.

In [ ]:
display(confusion_pairs(analysis).head(20))
messages = pd.DataFrame(
    [
        {"id": row["id"], "customer_message": row["state"]["customer_message"]}
        for row in analysis.cohort["examples"]
    ]
)
errors = analysis.outcomes.loc[
    ~analysis.outcomes["correct"],
    ["model_id", "id", "reference_label", "predicted_label", "status", "error", "latency_seconds"],
].merge(messages, on="id", validate="many_to_one")
with pd.option_context("display.max_colwidth", 100):
    display(errors.head(20))
display(
    analysis.outcomes.loc[
        analysis.outcomes["status"].ne("valid"), ["model_id", "id", "status", "error"]
    ]
)

## Export tables and figures

Export four CSV tables, five figures as vector PDF and 300-dpi PNG, and a manifest with source
checksums, analysis code checksum, and package versions. The original benchmark stays untouched.
Re-running replaces only the derived analysis files.

Use PDFs in the thesis and PNGs for previews. Reuse these functions for future runs to keep
metric definitions, colors, labels, and export settings consistent.

In [ ]:
export_directory = export_analysis(analysis, OUTPUT_DIRECTORY)
print(export_directory)
display(pd.DataFrame({"artifact": sorted(path.name for path in export_directory.iterdir())}))